In [1]:
# ======================================================
# README - pdnlS_2D Simulation Notebook OPTIMIZADO
# ======================================================
"""
Simulación 2D de la ecuación pdnlS_2D (parametrically driven nonlinear Schrödinger equation)
con integración RK4-FD, guardado de datos, y animaciones del módulo y fase.

REQUISITOS:
------------
pip install numpy scipy matplotlib

DEPENDENCIAS OPCIONALES:
-------------------------
- FFmpeg (solo si se desea guardar los videos como .mp4)
  Descargar desde https://ffmpeg.org/download.html
  y ajustar la ruta del ejecutable si es necesario:
      matplotlib.rcParams['animation.ffmpeg_path'] = r'C:\\ffmpeg\\bin\\ffmpeg.exe'

Si FFmpeg no está disponible, el código guardará animaciones en formato .gif automáticamente.

SALIDA:
-------
Se generan los siguientes archivos dentro de:
    D:/simulation_data/soliton_control_2D/alpha=.../beta=.../...

1. drive.png        → mapa espacial del bombeo
2. fields_data.npz  → grillas (x, y, t) y campos reales/imaginarios
3. module.mp4   → animación del módulo |A|
4. phase.mp4    → animación de la fase arg(A)

PARA LEER LOS DATOS GUARDADOS:
-------------------------------
data = np.load('fields_data.npz')
x = data['x_grid']
y = data['y_grid']
t = data['t_grid']
U = data['U_real'] + 1j * data['U_imag']

PARAMETROS CLAVE:
-----------------
alpha, beta, gamma_0, mu, nu, sigma   → controlan la ecuación
tf, dt, t_rate                        → control temporal
Lx, Ly, dx, dy                        → control espacial
"""

"\nSimulación 2D de la ecuación pdnlS_2D (parametrically driven nonlinear Schrödinger equation)\ncon integración RK4-FD, guardado de datos, y animaciones del módulo y fase.\n\nREQUISITOS:\n------------\npip install numpy scipy matplotlib\n\nDEPENDENCIAS OPCIONALES:\n-------------------------\n- FFmpeg (solo si se desea guardar los videos como .mp4)\n  Descargar desde https://ffmpeg.org/download.html\n  y ajustar la ruta del ejecutable si es necesario:\n      matplotlib.rcParams['animation.ffmpeg_path'] = r'C:\\ffmpeg\\bin\\ffmpeg.exe'\n\nSi FFmpeg no está disponible, el código guardará animaciones en formato .gif automáticamente.\n\nSALIDA:\n-------\nSe generan los siguientes archivos dentro de:\n    D:/simulation_data/soliton_control_2D/alpha=.../beta=.../...\n\n1. drive.png        → mapa espacial del bombeo\n2. fields_data.npz  → grillas (x, y, t) y campos reales/imaginarios\n3. module.mp4   → animación del módulo |A|\n4. phase.mp4    → animación de la fase arg(A)\n\nPARA LEER LOS DA

In [2]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation
import time
import datetime
import os
from numba import njit, prange
matplotlib.rcParams['animation.ffmpeg_path'] = r'C:\ffmpeg\bin\ffmpeg.exe'

# ------------------------------------
# FUNCIONES OPTIMIZADAS CON NUMBA
# ------------------------------------

@njit(parallel=True)
def compute_rhs_2d(U1, U2, alpha, beta, gamma1, gamma2, mu, nu, dx, dy):
    Nx, Ny = U1.shape
    F = np.empty((Nx, Ny), dtype=np.float64)
    G = np.empty((Nx, Ny), dtype=np.float64)
    dx2 = dx**2
    dy2 = dy**2

    for i in prange(Nx):
        for j in range(Ny):
            # Derivada segunda en X con Condición de Borde de Neumann
            if i == 0:
                d2U1_x = (U1[1, j] - U1[0, j]) / dx2
                d2U2_x = (U2[1, j] - U2[0, j]) / dx2
            elif i == Nx - 1:
                d2U1_x = (U1[Nx-2, j] - U1[Nx-1, j]) / dx2
                d2U2_x = (U2[Nx-2, j] - U2[Nx-1, j]) / dx2
            else:
                d2U1_x = (U1[i+1, j] - 2*U1[i, j] + U1[i-1, j]) / dx2
                d2U2_x = (U2[i+1, j] - 2*U2[i, j] + U2[i-1, j]) / dx2

            # Derivada segunda en Y con Condición de Borde de Neumann
            if j == 0:
                d2U1_y = (U1[i, 1] - U1[i, 0]) / dy2
                d2U2_y = (U2[i, 1] - U2[i, 0]) / dy2
            elif j == Ny - 1:
                d2U1_y = (U1[i, Ny-2] - U1[i, Ny-1]) / dy2
                d2U2_y = (U2[i, Ny-2] - U2[i, Ny-1]) / dy2
            else:
                d2U1_y = (U1[i, j+1] - 2*U1[i, j] + U1[i, j-1]) / dy2
                d2U2_y = (U2[i, j+1] - 2*U2[i, j] + U2[i, j-1]) / dy2

            lapU1 = d2U1_x + d2U1_y
            lapU2 = d2U2_x + d2U2_y

            u1_val = U1[i, j]
            u2_val = U2[i, j]
            g1_val = gamma1[i, j]
            g2_val = gamma2[i, j]

            mod_sq = u1_val**2 + u2_val**2

            F[i, j] = alpha * lapU2 + (beta * mod_sq + nu + g2_val) * u2_val + (g1_val - mu) * u1_val
            G[i, j] = -alpha * lapU1 - (beta * mod_sq + nu - g2_val) * u1_val - (g1_val + mu) * u2_val

    return F, G

@njit
def RK4_numba_2d(U1, U2, alpha, beta, gamma1, gamma2, mu, nu, dx, dy, dt, Nt, t_rate):
    Nx, Ny = U1.shape

    #  cuántos cuadros se guardarán para pre-asignar memoria
    num_saves = (Nt - 1) // t_rate + 1
    history_U1 = np.empty((num_saves, Nx, Ny), dtype=np.float64)
    history_U2 = np.empty((num_saves, Nx, Ny), dtype=np.float64)

    save_idx = 0


    for i in range(Nt):
        if i % t_rate == 0:
            history_U1[save_idx] = np.copy(U1)
            history_U2[save_idx] = np.copy(U2)
            save_idx += 1

        if i == Nt - 1:
            break

        k1_U1, k1_U2 = compute_rhs_2d(U1, U2, alpha, beta, gamma1, gamma2, mu, nu, dx, dy)

        U1_tmp = U1 + 0.5 * dt * k1_U1
        U2_tmp = U2 + 0.5 * dt * k1_U2
        k2_U1, k2_U2 = compute_rhs_2d(U1_tmp, U2_tmp, alpha, beta, gamma1, gamma2, mu, nu, dx, dy)

        U1_tmp = U1 + 0.5 * dt * k2_U1
        U2_tmp = U2 + 0.5 * dt * k2_U2
        k3_U1, k3_U2 = compute_rhs_2d(U1_tmp, U2_tmp, alpha, beta, gamma1, gamma2, mu, nu, dx, dy)

        U1_tmp = U1 + dt * k3_U1
        U2_tmp = U2 + dt * k3_U2
        k4_U1, k4_U2 = compute_rhs_2d(U1_tmp, U2_tmp, alpha, beta, gamma1, gamma2, mu, nu, dx, dy)

        U1 = U1 + (dt / 6.0) * (k1_U1 + 2*k2_U1 + 2*k3_U1 + k4_U1)
        U2 = U2 + (dt / 6.0) * (k1_U2 + 2*k2_U2 + 2*k3_U2 + k4_U2)

    return U1, U2, history_U1[:save_idx], history_U2[:save_idx]

In [32]:
# DIRECTORIOS Y PARAMETROS

project_name = '/soliton_control_2D'
disc = 'D:/'
eq = 'pdnlS_2D'

alpha = 1
beta = 1
gamma_0 = 0.14
nu = -0.068
sigma = 16
mu = 0.125

# DEFINIENDO GRILLA Y TIEMPOS
ti = 0
tf = 10000    #TIEMPO FINAL DE SIMULACIÓN
dt = 0.1
t_rate = 400
Lx = 130
Ly = 130
dx = 0.75
dy = 0.75

[tmin, tmax, dt] = [0, tf, dt]
[xmin, xmax, dx] = [- Lx / 2,  Lx / 2, dx]
[ymin, ymax, dy] = [- Ly / 2, Ly / 2, dy]

t_grid = np.arange(tmin, tmax + dt, dt)
x_grid = np.arange(xmin, xmax, dx)
y_grid = np.arange(ymin, ymax, dy)

T = tmax
Nt = t_grid.shape[0]
Nx = x_grid.shape[0]
Ny = y_grid.shape[0]

X, Y = np.meshgrid(x_grid, y_grid)
#Z = np.exp((- (X ** 2 + Y ** 2) / (2 * sigma ** 2)))    #SPACE DEPENDENT PUMP
Z = np.ones((Nx, Ny)) # HOMOGENEOUS PUMP

alpha_str = f"{alpha:.{4}f}"
beta_str = f"{beta:.{4}f}"
gamma_str = f"{gamma_0:.{4}f}"
nu_str = f"{nu:.{4}f}"
sigma_str = f"{sigma:.{4}f}"
mu_str = f"{mu:.{4}f}"

file = disc + '/simulation_data' + project_name
subfile = "/alpha=" + alpha_str + "/beta=" + beta_str + "/mu=" + mu_str + "/nu=" + nu_str + "/gamma=" + gamma_str + "/sigma=" + sigma_str
if not os.path.exists(file + subfile):
    os.makedirs(file + subfile)

# GRAFICANDO DRIVE
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
pcm1 = ax[0].pcolormesh(x_grid, y_grid, gamma_0 * np.real(Z), cmap="seismic", shading='auto')
cbar1 = plt.colorbar(pcm1, ax=ax[0], shrink=0.8)
cbar1.set_label('Re($Z$)', rotation=0, size=16, labelpad=-30, y=1.12)
ax[0].set_xlabel('$y$', size=16)
ax[0].set_ylabel('$x$', size=16)
ax[0].set_aspect('equal')

pcm2 = ax[1].pcolormesh(x_grid, y_grid, gamma_0 * np.imag(Z), cmap="seismic", shading='auto')
cbar2 = plt.colorbar(pcm2, ax=ax[1], shrink=0.8)
cbar2.set_label('Im($Z$)', rotation=0, size=16, labelpad=-30, y=1.12)
ax[1].set_xlabel('$y$', size=16)
ax[1].set_ylabel('$x$', size=16)
ax[1].set_aspect('equal')
plt.tight_layout()
plt.savefig(file + subfile + '/drive.png', dpi=300)
plt.close()

# Initial Conditions
delta = np.sqrt(- nu + np.sqrt(gamma_0 ** 2 - mu ** 2))
x_0 = 15
y_0 = 15
U_1_init = (np.sqrt(2) * delta / np.cosh(delta * (np.sqrt((X - x_0) ** 2 + (Y - y_0) ** 2) / np.sqrt(alpha)))) * np.real(np.exp(-1j * 0.5 * np.arccos(mu / gamma_0))) - (np.sqrt(2) * delta / np.cosh(delta * (np.sqrt((X + x_0) ** 2 + (Y + y_0) ** 2) / np.sqrt(alpha)))) * np.real(np.exp(-1j * 0.5 * np.arccos(mu / gamma_0)))
U_2_init = 0 * (np.sqrt(2) * delta / np.cosh(delta * (np.sqrt((X - x_0) ** 2 + (Y - y_0) ** 2) / np.sqrt(alpha)))) * np.imag(np.exp(-1j * 0.5 * np.arccos(mu / gamma_0))) - (np.sqrt(2) * delta / np.cosh(delta * (np.sqrt((X + x_0) ** 2 + (Y + y_0) ** 2) / np.sqrt(alpha)))) * np.real(np.exp(-1j * 0.5 * np.arccos(mu / gamma_0)))

# PARÁMETROS PREPARADOS PARA NUMBA
gamma1 = np.real(gamma_0 * Z)
gamma2 = np.imag(gamma_0 * Z)

In [ ]:
# Midiendo tiempo inicial
now = datetime.datetime.now()
print('Hora de Inicio: ' + str(now.hour) + ':' + str(now.minute) + ':' + str(now.second))
time_init = time.time()

# INTEGRACIÓN RK4 OPTIMIZADA
final_U1, final_U2, history_U1, history_U2 = RK4_numba_2d(
    U_1_init, U_2_init, alpha, beta, gamma1, gamma2, mu, nu, dx, dy, dt, Nt, t_rate
)

time_SIM = time.time()
print("Tiempo de Simulación: " + str(time_SIM - time_init) + ' seg')

# GUARDANDO DATOS
U_complex = history_U1 + 1j * history_U2
t_light = t_grid[::t_rate]
if len(t_light) > len(U_complex):
    t_light = t_light[:len(U_complex)]  # Ajuste por divisiones impares

modulo_light_1 = np.abs(U_complex)
arg_light_1 = np.angle(U_complex)
arg_light_1 = np.where(arg_light_1 > np.pi, arg_light_1 - 2 * np.pi, arg_light_1)
vmax_global = np.max(modulo_light_1)

output_data = os.path.join(file + subfile, 'fields_data.npz')
#np.savez_compressed(
#    output_data,
#    x_grid=x_grid,
#    y_grid=y_grid,
#    t_grid=t_light,
#    U_real=np.real(U_complex),
#    U_imag=np.imag(U_complex))

Hora de Inicio: 13:55:0


In [29]:
# === ANIMACIÓN DEL MÓDULO ============
frames = len(modulo_light_1)
skip = 1
frames_idx = range(0, frames, skip)
os.makedirs(file + subfile, exist_ok=True)
output_path_module = os.path.join(file + subfile, 'module.mp4')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(modulo_light_1[0], vmin=0, vmax=vmax_global, cmap="viridis", origin="lower", extent=[xmin, xmax, ymin, ymax], interpolation='nearest')
ax.set_xlabel('$y$', size=16)
ax.set_ylabel('$x$', size=16)
ax.grid(True, color='w', linestyle='--', linewidth=0.5, alpha=0.2)
cbar = fig.colorbar(im, ax=ax, shrink=0.9)
cbar.set_label('$|A|$', rotation=0, size=20, labelpad=-27, y=1.13)

def animate_mod(i):
    im.set_data(modulo_light_1[i])
    return [im]

anim_mod = FuncAnimation(fig, animate_mod, frames=frames_idx, interval=30, blit=False)
try:
    writervideo = animation.FFMpegWriter(fps=30, bitrate=3000)
    anim_mod.save(output_path_module, writer=writervideo, dpi=120)
except Exception as e:
    print(f"Error con FFmpeg (Módulo), guardando como gif temporal: {e}")
plt.close()

In [30]:
# === ANIMACIÓN DE LA FASE ============
output_path_phase = os.path.join(file + subfile, 'phase.mp4')
fig, ax = plt.subplots(figsize=(5, 4))
im2 = ax.imshow(arg_light_1[0], vmin=-np.pi, vmax=np.pi, cmap="hsv", origin="lower", extent=[xmin, xmax, ymin, ymax], interpolation='nearest')
ax.set_xlabel('$y$', size=16)
ax.set_ylabel('$x$', size=16)
ax.grid(True, color='w', linestyle='--', linewidth=0.5, alpha=0.2)
cbar2 = fig.colorbar(im2, ax=ax, shrink=0.9)
cbar2.set_label('$\\mathrm{arg}(A)$', rotation=0, size=20, labelpad=-27, y=1.13)

def animate_phase(i):
    im2.set_data(arg_light_1[i])
    return [im2]

anim_phase = FuncAnimation(fig, animate_phase, frames=frames_idx, interval=30, blit=False)
try:
    anim_phase.save(output_path_phase, writer=writervideo, dpi=120)
except Exception as e:
    print(f"Error con FFmpeg (Fase): {e}")
plt.close()

now = datetime.datetime.now()
print('Hora de Término: ' + str(now.hour) + ':' + str(now.minute) + ':' + str(now.second))
time_fin = time.time()
print("Tiempo Total del Script: " + str(time_fin - time_init) + ' seg')

Hora de Término: 13:54:7
Tiempo Total del Script: 31.225594997406006 seg
